# Proyecto Final Integrador — Grupo 9
**Diplomado UTN BA — Ciencia de Datos y Analisis Avanzado**

Tamara Iribarren · Santiago Alvarez Peratta · Julio Aguiar · Jose Portillo · Agustin Juarez

**Problema:** Prediccion de demoras en entregas logisticas

**Metodologia:** CRISP-DM

## 0. Instalacion de librerias

In [ ]:
!pip install pandas openpyxl scikit-learn matplotlib seaborn joblib --quiet

## 1. Carga del dataset

In [ ]:
from google.colab import files
import io, pandas as pd

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_raw = pd.read_excel(
    io.BytesIO(uploaded[filename]),
    dtype={"shipping_distance_km": str, "processing_time_hours": str}
)
print(f"Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}")
df_raw.head()

## 2. Limpieza de datos (Fase 3 CRISP-DM)

In [ ]:
import numpy as np

df = df_raw.copy()

def contar_no_numericos(serie):
    return pd.to_numeric(serie, errors="coerce").isna().sum()

print("Valores corruptos antes de limpiar:")
print(f"  shipping_distance_km  : {contar_no_numericos(df['shipping_distance_km'])}")
print(f"  processing_time_hours : {contar_no_numericos(df['processing_time_hours'])}")

df["shipping_distance_km"]  = pd.to_numeric(df["shipping_distance_km"],  errors="coerce")
df["processing_time_hours"] = pd.to_numeric(df["processing_time_hours"], errors="coerce")
df["shipping_distance_km"].fillna(df["shipping_distance_km"].median(), inplace=True)
df["processing_time_hours"].fillna(df["processing_time_hours"].median(), inplace=True)

print(f"\nNulos restantes: {df.isnull().sum().sum()}")
print(f"Tasa de retraso global: {df['delayed'].mean():.1%}")
print("Dataset limpio OK")

## 3. EDA — Analisis Exploratorio

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["shipping_method", "weather_condition", "order_priority"]):
    tasa = df.groupby(col)["delayed"].mean().sort_values(ascending=False)
    sns.barplot(x=tasa.index, y=tasa.values, ax=ax, palette="coolwarm")
    ax.set_title(f"Retraso por {col}")
    ax.set_ylim(0, 0.45)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.tight_layout(); plt.show()

In [ ]:
combo = df.groupby(["weather_condition","shipping_method"])["delayed"].mean().reset_index()
combo.columns = ["weather","method","rate"]
combo = combo.sort_values("rate", ascending=False)
etiquetas = combo["weather"] + " + " + combo["method"]
colores = ["#E24B4A" if r > 0.33 else "#BA7517" if r > 0.30 else "#378ADD" for r in combo["rate"]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(etiquetas[::-1], combo["rate"][::-1], color=colores[::-1])
ax.axvline(df["delayed"].mean(), color="black", linestyle="--", linewidth=1.2,
           label=f"Media global ({df['delayed'].mean():.1%})")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Combinaciones de riesgo: clima x metodo de envio")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
df["month"] = df["order_date"].dt.month
monthly = df.groupby("month")["delayed"].mean()
nombres = {1:"Enero", 2:"Febrero", 3:"Marzo", 4:"Abril"}
monthly.index = monthly.index.map(nombres)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monthly.index, monthly.values, marker="o", color="#E24B4A", linewidth=2.5)
ax.fill_between(monthly.index, monthly.values, alpha=0.1, color="#E24B4A")
ax.set_title("Tasa de retraso mensual")
ax.set_ylim(0.24, 0.33)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
plt.tight_layout(); plt.show()

## 4. Feature Engineering — 4 experimentos

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Experimento 1: variables temporales (hipotesis: retrasos en ciertos dias)
df["day_of_week"] = df["order_date"].dt.dayofweek
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

# Experimento 2: agrupacion climatica Favorable/Adversa
df["bad_weather"]  = df["weather_condition"].isin(["Storm","Fog"]).astype(int)
df["good_weather"] = (df["weather_condition"] == "Clear").astype(int)
df["is_storm"]     = (df["weather_condition"] == "Storm").astype(int)
df["is_fog"]       = (df["weather_condition"] == "Fog").astype(int)

# Interacciones detectadas en EDA
df["is_air"]        = (df["shipping_method"] == "Air").astype(int)
df["risk_combo"]    = df["bad_weather"] * df["is_air"]
df["low_proc_time"] = (df["processing_time_hours"] < 16).astype(int)
df["inv_per_qty"]   = df["warehouse_inventory_level"] / (df["order_quantity"] + 1)

encoders = {}
for col in ["shipping_method", "weather_condition", "order_priority"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

FEATURES = [
    "supplier_reliability_score", "warehouse_inventory_level", "order_quantity",
    "shipping_distance_km", "shipping_method", "weather_condition",
    "processing_time_hours", "order_priority",
    "month", "day_of_week", "is_weekend",
    "is_storm", "is_fog", "bad_weather", "good_weather",
    "is_air", "risk_combo", "low_proc_time", "inv_per_qty",
]
X = df[FEATURES]; y = df["delayed"]
print(f"Features totales: {len(FEATURES)}")
print(f"Clases — No retrasado: {(y==0).sum()} | Retrasado: {(y==1).sum()}")

## 5. Experimento 3 — Comparativa de modelos

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
import warnings; warnings.filterwarnings("ignore")

# Split 80/20 estratificado (segun pre-entrega)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# HistGradientBoosting = equivalente funcional a XGBoost en scikit-learn
candidatos = {
    "Logistic Regression (baseline)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", C=0.1, max_iter=1000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", max_depth=8, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42),
    "XGBoost (HistGBM-equiv)": HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_depth=4, class_weight="balanced", random_state=42),
}

print(f"{'Modelo':<35} {'AUC':>7} {'F1':>7} {'Recall':>8} {'Prec':>8}")
print("-"*68)
resultados = {}
for nombre, modelo in candidatos.items():
    auc = cross_val_score(modelo, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    f1  = cross_val_score(modelo, X_train, y_train, cv=cv, scoring="f1").mean()
    rec = cross_val_score(modelo, X_train, y_train, cv=cv, scoring="recall").mean()
    pre = cross_val_score(modelo, X_train, y_train, cv=cv, scoring="precision").mean()
    resultados[nombre] = dict(auc=auc, f1=f1, recall=rec, precision=pre)
    print(f"{nombre:<35} {auc:>7.4f} {f1:>7.4f} {rec:>8.4f} {pre:>8.4f}")

mejor_nombre = max(resultados, key=lambda k: resultados[k]["f1"])
print(f"\nMejor modelo por F1: {mejor_nombre}")

In [ ]:
mejor_modelo = candidatos[mejor_nombre]
mejor_modelo.fit(X_train, y_train)
y_pred = mejor_modelo.predict(X_test)
y_prob = mejor_modelo.predict_proba(X_test)[:, 1]

print(f"=== {mejor_nombre} — Test Set ===")
print(classification_report(y_test, y_pred, target_names=["No retrasado","Retrasado"]))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["No retrasado","Retrasado"],
    cmap="Blues", ax=axes[0])
axes[0].set_title("Matriz de confusion — Test Set")

nombres = list(resultados.keys())
aucs = [resultados[n]["auc"] for n in nombres]
axes[1].barh(nombres, aucs, color="#378ADD")
axes[1].axvline(0.5, color="red", linestyle="--", linewidth=1, label="Baseline aleatorio")
axes[1].set_title("AUC-ROC comparativo"); axes[1].legend()
axes[1].set_xlim(0.44, 0.60)
plt.tight_layout(); plt.show()

## 6. Experimento 4 — Red Neuronal MLP (proxy de GRU)

In [ ]:
from sklearn.neural_network import MLPClassifier

# Split TEMPORAL: orden cronologico para respetar estructura temporal
df_sorted = df.sort_values("order_date").reset_index(drop=True)
X_seq = df_sorted[FEATURES]; y_seq = df_sorted["delayed"]
split_idx = int(len(X_seq) * 0.8)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_seq.iloc[:split_idx])
X_te_s = sc.transform(X_seq.iloc[split_idx:])
y_tr_seq = y_seq.iloc[:split_idx]; y_te_seq = y_seq.iloc[split_idx:]

# Arquitectura 64->32->16 con regularizacion L2 y early stopping
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16), activation="relu", solver="adam",
    alpha=0.01, learning_rate_init=0.001, max_iter=300, random_state=42,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=20,
)
mlp.fit(X_tr_s, y_tr_seq)
yp_mlp    = mlp.predict(X_te_s)
yprob_mlp = mlp.predict_proba(X_te_s)[:, 1]

print("=== Red Neuronal MLP — Split temporal ===")
print(classification_report(y_te_seq, yp_mlp, target_names=["No retrasado","Retrasado"]))
print(f"AUC-ROC: {roc_auc_score(y_te_seq, yprob_mlp):.4f} | Epocas: {mlp.n_iter_}")

## 7. Diagnostico y conclusiones

In [ ]:
print('''
DIAGNOSTICO:
Todos los modelos obtienen AUC ~= 0.50 (equivalente al azar).
El problema no es el algoritmo, es la falta de senal en los datos.

El dataset captura QUE se pidio, pero no COMO se ejecuto la entrega.

Para mejorar el modelo, la empresa deberia capturar:
  1. Historial de puntualidad por proveedor
  2. Ocupacion real del almacen (inventario / capacidad)
  3. Tiempo real de cada etapa (preparacion, despacho, transito)
  4. Metricas climaticas cuantitativas por ruta
  5. Tasa historica de retraso por destino/ruta
''')

## 8. Guardar y descargar

In [ ]:
import joblib
from google.colab import files

joblib.dump({"modelo": mejor_modelo, "encoders": encoders, "features": FEATURES},
            "modelo_delay_grupo9.pkl")
df.to_excel("supply_chain_limpio.xlsx", index=False)
print("Descargando...")
files.download("modelo_delay_grupo9.pkl")
files.download("supply_chain_limpio.xlsx")